In [1]:
from dotenv import load_dotenv

from langchain_teddynote import logging
from langchain_openai import ChatOpenAI
from langchain_teddynote.messages import stream_response
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

from pydantic import BaseModel, Field
from itertools import chain

In [2]:
load_dotenv()

True

In [3]:
logging.langsmith("langchain-03")

LangSmith 추적을 시작합니다.
[프로젝트명]
langchain-03


In [4]:
llm = ChatOpenAI(
    temperature=0, 
    model_name="gpt-4.1-mini"
)

In [5]:
email_conversation = """From: 김철수 (chulsoo.kim@bikecorporation.me)
To: 이은채 (eunchae@teddyinternational.me)
Subject: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안

안녕하세요, 이은채 대리님,

저는 바이크코퍼레이션의 김철수 상무입니다. 최근 보도자료를 통해 귀사의 신규 자전거 "ZENESIS"에 대해 알게 되었습니다. 바이크코퍼레이션은 자전거 제조 및 유통 분야에서 혁신과 품질을 선도하는 기업으로, 이 분야에서의 장기적인 경험과 전문성을 가지고 있습니다.

ZENESIS 모델에 대한 상세한 브로슈어를 요청드립니다. 특히 기술 사양, 배터리 성능, 그리고 디자인 측면에 대한 정보가 필요합니다. 이를 통해 저희가 제안할 유통 전략과 마케팅 계획을 보다 구체화할 수 있을 것입니다.

또한, 협력 가능성을 더 깊이 논의하기 위해 다음 주 화요일(1월 15일) 오전 10시에 미팅을 제안합니다. 귀사 사무실에서 만나 이야기를 나눌 수 있을까요?

감사합니다.

김철수
상무이사
바이크코퍼레이션
"""

출력 파서를 사용하지 않는 경우

In [6]:
prompt = PromptTemplate.from_template("다음의 이메일 내용중 중요한 내용을 추출해 주세요.\n\n{email_conversation}")

chain = prompt | llm

answer = chain.stream({"email_conversation": email_conversation})

In [7]:
print(stream_response(answer, return_output=True))

다음은 이메일의 중요한 내용입니다:

- 발신자: 김철수 상무(바이크코퍼레이션)
- 수신자: 이은채 대리(테디인터내셔널)
- 목적: "ZENESIS" 자전거 유통 협력 논의 및 미팅 일정 제안
- 요청 사항: ZENESIS 자전거의 상세 브로슈어(기술 사양, 배터리 성능, 디자인 정보)
- 미팅 제안: 1월 15일 화요일 오전 10시, 귀사 사무실에서 미팅 희망
- 배경: 바이크코퍼레이션은 자전거 제조 및 유통 분야에서 경험과 전문성을 보유하고 있음다음은 이메일의 중요한 내용입니다:

- 발신자: 김철수 상무(바이크코퍼레이션)
- 수신자: 이은채 대리(테디인터내셔널)
- 목적: "ZENESIS" 자전거 유통 협력 논의 및 미팅 일정 제안
- 요청 사항: ZENESIS 자전거의 상세 브로슈어(기술 사양, 배터리 성능, 디자인 정보)
- 미팅 제안: 1월 15일 화요일 오전 10시, 귀사 사무실에서 미팅 희망
- 배경: 바이크코퍼레이션은 자전거 제조 및 유통 분야에서 경험과 전문성을 보유하고 있음


Pydantic 스타일로 파싱하면..

In [8]:
class EmailSummary(BaseModel):
    person: str = Field(description="메일을 보낸 사람")
    email: str = Field(description="메일을 보낸 사람의 이메일 주소")
    subject: str = Field(description="메일 제목")
    summary: str = Field(description="메일 본문을 요약한 텍스트")
    date: str = Field(description="메일 본문에 언급된 미팅 날짜와 시간")

parser = PydanticOutputParser(pydantic_object=EmailSummary)

In [9]:
print(parser.get_format_instructions())  # instruction 출력

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "email": {"description": "메일을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 미팅 날짜와 시간", "title": "Date", "type": "string"}}, "required": ["person", "email", "subject", "summary", "date"]}
```


In [10]:
prompt = PromptTemplate.from_template(
    """
You are a helpful assistant. Please answer the following questions in KOREAN.

QUESTION:
{question}

EMAIL CONVERSATION:
{email_conversation}

FORMAT:
{format}
"""
)

# question: user의 질문
# email_conversation: 이메일 본문의 내용
# format: 형식

In [11]:
prompt = prompt.partial(format=parser.get_format_instructions())

In [12]:
chain = prompt | llm

In [13]:
response = chain.stream(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용중 주요 내용을 추출해 주세요."
    }
)

output = stream_response(response, return_output=True)

```json
{
  "person": "김철수",
  "email": "chulsoo.kim@bikecorporation.me",
  "subject": "\"ZENESIS\" 자전거 유통 협력 및 미팅 일정 제안",
  "summary": "바이크코퍼레이션 김철수 상무가 ZENESIS 자전거의 상세 브로슈어(기술 사양, 배터리 성능, 디자인)를 요청하며, 유통 전략과 마케팅 계획 수립을 위해 협력 가능성을 논의하고자 1월 15일 화요일 오전 10시에 미팅을 제안함.",
  "date": "1월 15일 화요일 오전 10시"
}
```

In [14]:
# 결과 파싱
structured_output = parser.parse(output)

print(structured_output)

person='김철수' email='chulsoo.kim@bikecorporation.me' subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안' summary='바이크코퍼레이션 김철수 상무가 ZENESIS 자전거의 상세 브로슈어(기술 사양, 배터리 성능, 디자인)를 요청하며, 유통 전략과 마케팅 계획 수립을 위해 협력 가능성을 논의하고자 1월 15일 화요일 오전 10시에 미팅을 제안함.' date='1월 15일 화요일 오전 10시'


In [15]:
chain = prompt | llm | parser  # parser가 추가된 체인

In [16]:
response = chain.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용중 주요 내용을 추출해 주세요.",
    }
)

response

EmailSummary(person='김철수', email='chulsoo.kim@bikecorporation.me', subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안', summary='바이크코퍼레이션 김철수 상무가 ZENESIS 자전거에 대한 상세 브로슈어(기술 사양, 배터리 성능, 디자인)를 요청하고, 유통 전략과 마케팅 계획 수립을 위해 협력 가능성을 논의하고자 1월 15일 화요일 오전 10시에 미팅을 제안함.', date='1월 15일 화요일 오전 10시')

with_structured_output()

In [17]:
llm_structured = ChatOpenAI(temperature=0, model_name="gpt-4.1-mini").with_structured_output(EmailSummary)

In [18]:
answer_structured = llm_structured.invoke(email_conversation)  # with_structured_output 함수는 stream 기능 미지원

In [19]:
answer_structured

EmailSummary(person='김철수', email='chulsoo.kim@bikecorporation.me', subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안', summary='바이크코퍼레이션의 김철수 상무가 이은채 대리에게 ZENESIS 자전거에 대한 상세 브로슈어 요청과 함께, 기술 사양, 배터리 성능, 디자인 정보가 필요하다고 전달했습니다. 또한, 1월 15일 화요일 오전 10시에 미팅을 제안하며 협력 가능성을 논의하고자 합니다.', date='2024-01-08')